WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

Thread a clamped rod through a catheter tube driven by wall contact alone.

The tube and its lumen axis come from `carotid_reader_catheter.ipynb`, in m, with the
mouth at the origin and insertion along +x. The rod is fixed at its proximal end and
the tube slides back over it, so the free tip enters the mouth and is progressively
forced along the track. Relative motion makes this identical to advancing the rod,
while keeping the displacement field as pure deformation rather than deformation plus
a rigid advance.

In [ ]:
import dataclasses

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.spatial as spy_spatial
import polyscope as ps
import trimesh

import crops.src.bc as _bc
import crops.src.contact as _contact
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.high_res_plot_settings)

### Track

In [ ]:
# CCA -> ICA (line 0), tube radius matched to the ICA. The earlier multi-branch
# track is catheter_k_002_left.stl / catheter_k_002_left_0_1_2_3_4_5.csv.
pipe = trimesh.load_mesh("crops/data/obstacles/catheter_k_002_left_0.stl")
if isinstance(pipe, trimesh.Scene):
  pipe = trimesh.util.concatenate(pipe.dump())
target = pd.read_csv("crops/data/targets/catheter_k_002_left_0.csv")
target_xyz = target[["x", "y", "z"]].to_numpy()
track_length = float(target["s"].iloc[-1])

# The mouth ring lies exactly in x = 0, so its vertices give the two tube radii.
mouth_ring = np.linalg.norm(pipe.vertices[np.abs(pipe.vertices[:, 0]) < 1e-9, 1:], axis=-1)
bore_radius, outer_radius = mouth_ring.min(), mouth_ring.max()
wall_thickness = outer_radius - bore_radius
print(f"watertight {pipe.is_watertight}, extents {np.round(pipe.extents, 4)} m")
print(f"bore radius {bore_radius:.4f} m, wall {wall_thickness:.4f} m, track {track_length:.4f} m")

### Rod

In [ ]:
# A little longer than the track so the tip leaves the outlet at full insertion.
# Wall contact acts at element centres, so elements are kept shorter than the
# contact radius, otherwise a centre can straddle the thin wall.
entry_gap = 0.005
rod_length = track_length + 2.0 * entry_gap
num_elems = int(np.ceil(rod_length / 0.75e-3))
num_nodes = num_elems + 1
shear_modulus = 1.0e6

fiber_offset = 0.0005
fiber_radius = 0.0003
offsets_local = np.array(
  [[fiber_offset, 0.0], [-fiber_offset, 0.0],
   [0.0, fiber_offset], [0.0, -fiber_offset]]
)
num_fibers = 4
area_per_fiber = np.pi * fiber_radius**2

# A deterministic half-wave bow breaks the symmetry of the threading problem, which
# would otherwise sit on a bifurcation the moment the tip meets the fillet. The
# amplitude is small enough that the reference is effectively straight.
initial_bow_ratio = 1.0e-3

rod = _mesher.build_rod_geometry(
  analytical_curve=lambda s: jnp.array([s, 0.0, 0.0]),
  parametric_coords=jnp.linspace(0.0, rod_length, num_nodes),
  fiber_offsets=jnp.asarray(np.tile(offsets_local, (num_elems, 1, 1))),
  fiber_areas=jnp.full((num_elems, num_fibers), area_per_fiber),
  fiber_mass_density=jnp.ones((num_elems, num_fibers)),
  fiber_centroid_moi=jnp.zeros((num_elems, num_fibers, 2)),
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
  initial_bow_ratio=initial_bow_ratio,
)
contact_radii = np.asarray(rod.contact_radii)
bending_stiffness = (
  3.0 * shear_modulus * np.sum(area_per_fiber * offsets_local[:, 0] ** 2)
)
print(f"contact radius = {contact_radii.max():.4f} m")
print(f"EI             = {bending_stiffness:.4e} N m^2")

### Obstacle

In [ ]:
print(f"clearance = {bore_radius - contact_radii.max():.4f} m")

In [ ]:
# Padding at least one contact radius puts the grid boundary where the contact force
# is already identically zero, so the discontinuity across it is invisible to the
# solver. Resolution is set by the wall, the thinnest feature: at two voxels across
# it the wall reads as solid everywhere, while a coarser grid leaves holes in it.
padding = 3.0 * contact_radii.max()
voxel = 0.5 * wall_thickness
grid_shape = tuple(int(n) for n in np.ceil((pipe.extents + 2.0 * padding) / voxel))
wall_penalty_stiffness = 1.0e6

obstacle = _contact.Obstacle.from_mesh(
  mesh=pipe,
  grid_shape=grid_shape,
  wall_penalty_stiffness=wall_penalty_stiffness,
  padding=padding,
  extrap_fill_value=10.0,
)
spacing = np.asarray(obstacle.spacing)
print(f"grid {grid_shape}, spacing {np.round(spacing, 5)} m")
print(f"cells across the bore = {2.0 * bore_radius / spacing.max():.1f}  (want > 10)")
print(f"cells across the wall = {wall_thickness / spacing.max():.1f}  (want >= 2)")

In [ ]:
# The pipe mouth sits at the origin of its own body frame, so translating that frame
# along x positions the mouth in the lab. Starting just beyond the free tip and
# advancing toward the clamp draws the pipe back over the rod, threading the tip
# deeper with every step. The mouth stops short of the clamp: the track leaves the
# x axis within a few mm, so any further and the clamped end would sit in the wall.
max_advance_per_step = 1.0e-3
mouth_start = rod_length + entry_gap
mouth_end = entry_gap
num_steps = int(np.ceil((mouth_start - mouth_end) / max_advance_per_step))
mouth_positions = np.linspace(mouth_start, mouth_end, num_steps)

movement = _contact.ObstacleMovement.from_translations(
  jnp.stack(
    [jnp.asarray(mouth_positions), jnp.zeros(num_steps), jnp.zeros(num_steps)],
    axis=-1,
  )
)
obstacle = dataclasses.replace(obstacle, movement=movement)
print(f"mouth travels {mouth_start:.3f} -> {mouth_end:.3f} m over {num_steps} steps")
print(f"deepest insertion = {rod_length - mouth_end:.3f} m along the pipe "
      f"(track {track_length:.3f} m)")

In [ ]:
# Sanity check the pose map before solving. The target is the lumen axis, so it should
# read the bore radius at every step, and the clamp must stay in free space, clear of
# the contact band, for the whole insertion.
for step in [0, num_steps // 2, num_steps - 1]:
  obj_trans, obj_rot_mat = obstacle.pose_at_step(step)
  axis_lab = jnp.asarray(target_xyz) @ obj_rot_mat.T + obj_trans
  phi_axis = np.asarray(obstacle.compute_sdf(axis_lab, obj_trans, obj_rot_mat))
  phi_clamp = float(obstacle.compute_sdf(jnp.zeros((1, 3)), obj_trans, obj_rot_mat)[0])
  print(f"step {step:3d}  mouth x = {float(obj_trans[0]):.4f}  "
        f"phi on axis {phi_axis.min():.5f}..{phi_axis.max():.5f} (bore {bore_radius:.5f})  "
        f"phi at clamp {phi_clamp:+.5f}")

### Boundary conditions and problem

In [ ]:
clamped_dofs = (
  _utils.Translation.X, _utils.Translation.Y, _utils.Translation.Z,
  _utils.Rotation.THETA_X, _utils.Rotation.THETA_Y, _utils.Rotation.THETA_Z,
)
bc = _bc.process_boundary_conditions(
  rod,
  [
    _bc.DirichletBC(
      values=(dof, [np.array([0.0]) for _ in range(num_steps)]),
      node_targets=[np.array([0]) for _ in range(num_steps)],
      name=f"clamp_{dof}",
    )
    for dof in clamped_dofs
  ],
)
nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings={
    "max_iter": 200, "threshold": 1e-6, "abs_threshold": 1e-6, "lam_min": 0.01,
    "line_search_max_iter": 15, "line_search_shrink": 0.5,
    "line_search_alpha_min": 1e-6, "line_search_armijo_c": 1e-4,
  },
  linear_settings={"solver": _solver.LinearSolvers.SCIPY_SPARSE, "rtol": 1e-10},
)
problem = _fe.CosseratProblem(
  rod=rod, bc=bc, nr_solver=nr_solver, obstacle=obstacle
)

### Contact forces

`compute_contact_forces` belongs on `CosseratProblem`. It is defined here so the
notebook runs as-is; move it and delete this cell.

In [ ]:
def compute_contact_forces(problem, u: jnp.ndarray, bc_step_idx: int = 0):
  """Nodal contact force exerted by the obstacle on the rod.

  The quantity assembled inside the residual is the positive energy gradient, which
  acts as an internal force and is subtracted from the residual. The physical force
  applied to the rod is its negative, which is what is returned, so a positive
  component points in the direction the obstacle pushes.

  Only translational components are returned. The wall contact energy depends on the
  element centre position, which is the average of the two nodal translations, so
  the rotational components of the assembled vector are identically zero.

  Args:
    problem: `CosseratProblem` carrying the rod and the obstacle.
    u: Array of shape `(num_dofs,)` [m, rad] of nodal displacements and rotations.
    bc_step_idx: Load step index used to look up the obstacle pose.

  Returns:
    Array of shape `(num_nodes, num_translational_dofs_per_node)` [N].
  """
  obj_trans, obj_rot_mat = problem.obstacle.pose_at_step(bc_step_idx)
  f_contact, _ = _contact.compute_wall_contact_forces_and_tangent(
    u_elems=u[problem.rod.elem_dof_mat],
    rod=problem.rod,
    obstacle=problem.obstacle,
    compute_jacobian=False,
    obj_trans=obj_trans,
    obj_rot_mat=obj_rot_mat,
  )
  num_trans = problem.rod.num_translational_dofs_per_node
  return -f_contact.reshape(problem.rod.num_nodes, -1)[:, :num_trans]

### Solve

In [ ]:
u_star, u_history = problem.solve(
  u0=jnp.zeros(rod.num_dofs),
  shear_moduli=jnp.full((num_elems, num_fibers), shear_modulus),
)
history = np.asarray(u_history).reshape(num_steps, num_nodes, -1)

contact_forces = np.asarray(
  jax.vmap(lambda u, k: compute_contact_forces(problem, u, k))(
    jnp.asarray(u_history), jnp.arange(num_steps)
  )
)
force_magnitude = np.linalg.norm(contact_forces, axis=-1)
print(f"peak nodal contact force = {force_magnitude.max():.4f} N")
print(f"max nodal rotation       = "
      f"{np.degrees(np.abs(history[..., 3:]).max()):.1f} deg   (2 pi wall at 360)")

### Contact forces

In [ ]:
node_arc = np.asarray(rod.arc_coords)
fig, (ax_dist, ax_total) = plt.subplots(1, 2, figsize=(12, 4.5))
for step in range(0, num_steps, max(1, num_steps // 6)):
  ax_dist.plot(node_arc, force_magnitude[step], lw=1.2,
               label=f"mouth {mouth_positions[step]:.3f}")
ax_dist.set_xlabel(r"reference arc coordinate $\hat{s}$ [m]")
ax_dist.set_ylabel("nodal contact force [N]")
ax_dist.legend(fontsize=7)

ax_total.plot(mouth_positions, force_magnitude.sum(axis=1), "o-", ms=3, label="total")
ax_total.plot(mouth_positions, force_magnitude.max(axis=1), "s-", ms=3, label="peak")
ax_total.invert_xaxis()
ax_total.set_xlabel("pipe mouth position [m]")
ax_total.set_ylabel("contact force [N]")
ax_total.legend()
fig.tight_layout()
plt.show()

### Shapes

In [ ]:
# The track is 3D, so the shapes are drawn in 3D against the lumen axis at each pose.
ax = plt.figure(figsize=(8, 6)).add_subplot(projection="3d")
for step in np.linspace(0, num_steps - 1, 6, dtype=int):
  shape = np.asarray(rod.node_coords) + history[step, :, :3]
  (line,) = ax.plot(*shape.T, lw=1.4, label=f"mouth {mouth_positions[step]:.3f}")
final_axis = target_xyz + np.array([mouth_positions[-1], 0.0, 0.0])
ax.plot(*final_axis.T, "k--", lw=1.0, label="lumen axis (final)")
ax.set_aspect("equal")
ax.set_xlabel("$x$ [m]")
ax.set_ylabel("$y$ [m]")
ax.set_zlabel("$z$ [m]")
ax.legend(fontsize=7)
plt.show()

### Polyscope

In [ ]:
final_step = num_steps - 1
coords, directors, fiber_coords = (
  _mesher.extract_coords_and_director_from_deformation(
    rod, jnp.asarray(history[final_step].ravel())
  )
)
# Nodal forces averaged onto elements, which is what a curve network colours by.
elem_force = 0.5 * (
  force_magnitude[final_step, :-1] + force_magnitude[final_step, 1:]
)
node_radii = np.empty(num_nodes)
node_radii[0], node_radii[-1] = contact_radii[0], contact_radii[-1]
node_radii[1:-1] = 0.5 * (contact_radii[:-1] + contact_radii[1:])

_viz.init_scene_ps()
_viz.plot_rod_centerline_ps(
  np.asarray(coords),
  node_radii=node_radii,
  elem_field=elem_force,
  elem_field_val_range=(0.0, float(force_magnitude.max())),
  cmap="viridis",
  name="rod",
)
_viz.plot_obstacle_ps(obstacle, step_idx=final_step, name="pipe")
ps.show()

### Verification

A rod that stays in the lumen keeps its element centres within `bore - contact radius` of
the lumen axis, and its element centres never read a signed distance much below the
contact radius. Both are geometric statements, independent of the material, so a
violation means the tip has tunnelled through the wall or the solve has failed.

In [ ]:
allowed_offset = bore_radius - contact_radii.max()
axis_tree = spy_spatial.cKDTree(target_xyz)
offset, worst_phi = np.zeros((num_steps, num_elems)), np.zeros(num_steps)
for step in range(num_steps):
  # Element centres, since that is where the wall contact acts; a tip node pressed
  # against the wall can sit half an element further out than its centre.
  shape = np.asarray(rod.node_coords) + history[step, :, :3]
  centres = 0.5 * (shape[:-1] + shape[1:])
  obj_trans, obj_rot_mat = obstacle.pose_at_step(step)
  in_body = (centres - np.asarray(obj_trans)) @ np.asarray(obj_rot_mat)
  distance, nearest = axis_tree.query(in_body)
  inside = (in_body[:, 0] > 0.0) & (nearest < len(target_xyz) - 1)
  offset[step] = np.where(inside, distance, np.nan)
  worst_phi[step] = float(obstacle.compute_sdf(jnp.asarray(centres), obj_trans, obj_rot_mat).min())

fig, (ax_offset, ax_phi) = plt.subplots(1, 2, figsize=(12, 4.5))
elem_arc = 0.5 * (node_arc[:-1] + node_arc[1:])
ax_offset.plot(elem_arc, offset[-1] * 1e3, lw=1.5, label="final")
ax_offset.plot(elem_arc, np.nanmax(offset, axis=0) * 1e3, lw=1.0, label="max over steps")
ax_offset.axhline(allowed_offset * 1e3, color="k", ls="--", lw=1.2, label="bore - contact radius")
ax_offset.set_xlabel(r"reference arc coordinate $\hat{s}$ [m]")
ax_offset.set_ylabel("offset from lumen axis [mm]")
ax_offset.legend()
ax_phi.plot(mouth_positions, worst_phi * 1e3, "o-", ms=3)
ax_phi.axhline(contact_radii.max() * 1e3, color="k", ls="--", lw=1.2, label="contact radius")
ax_phi.invert_xaxis()
ax_phi.set_xlabel("pipe mouth position [m]")
ax_phi.set_ylabel(r"min $\phi$ at element centres [mm]")
ax_phi.legend()
fig.tight_layout()
plt.show()

# The penalty lets the centres sit slightly inside the contact band, so the offset may
# exceed the geometric limit by that penetration and no more.
penetration = contact_radii.max() - worst_phi.min()
escaped = np.flatnonzero(np.nanmax(np.nan_to_num(offset), axis=1) > allowed_offset + penetration)
print(f"max offset from lumen axis = {np.nanmax(offset) * 1e3:.3f} mm "
      f"(allowed {allowed_offset * 1e3:.3f} + {penetration * 1e3:.3f} mm penetration)")
print(f"deepest penetration        = {penetration * 1e3:.3f} mm into the contact band")
print(f"steps with the rod outside the lumen: {escaped.tolist()}")

In [ ]:
import dataclasses

# Registering the obstacle with its schedule stripped draws it at the identity pose,
# which is its body frame. Everything else is then transformed into that frame, so
# the pipe is stationary and the rod is what moves.
static_pipe = dataclasses.replace(obstacle, movement=None)

force_range = (0.0, float(force_magnitude.max()))
node_radii = np.empty(num_nodes)
node_radii[0], node_radii[-1] = contact_radii[0], contact_radii[-1]
node_radii[1:-1] = 0.5 * (contact_radii[:-1] + contact_radii[1:])

_viz.init_scene_ps()
_viz.plot_obstacle_ps(static_pipe, step_idx=0, name="pipe")
_viz.plot_rod_centerline_ps(
  np.zeros((num_nodes, 3)),
  node_radii=node_radii,
  elem_field=np.zeros(num_elems),
  elem_field_val_range=force_range,
  cmap="viridis",
  name="rod",
)

# Fixed once, so Polyscope does not re-fit the view as the rod sweeps through.
pipe_center = pipe.bounds.mean(axis=0)
ps.look_at(
  camera_location=tuple(pipe_center + 1.5 * pipe.extents.max() * np.array([1.0, 0.6, 1.0])),
  target=tuple(pipe_center),
)

In [ ]:
import imageio.v2 as imageio
import pathlib

frame_dir = pathlib.Path("frames")
frame_dir.mkdir(exist_ok=True)

for step in range(num_steps):
  coords, _, _ = _mesher.extract_coords_and_director_from_deformation(
    rod, jnp.asarray(history[step].ravel())
  )
  obj_trans, obj_rot_mat = obstacle.pose_at_step(step)
  # Lab frame to pipe body frame. The pipe is drawn fixed, so the rod carries the
  # relative motion and the animation reads as an insertion.
  coords_in_pipe = _utils.affine_transform(
    coords, obj_trans, obj_rot_mat, "GLOBAL_TO_LOCAL"
  )
  elem_force = 0.5 * (force_magnitude[step, :-1] + force_magnitude[step, 1:])

  _viz.update_rod_centerline_ps(
    np.asarray(coords_in_pipe), elem_field=elem_force, name="rod"
  )
  ps.screenshot(str(frame_dir / f"frame_{step:03d}.png"), transparent_bg=False)

imageio.mimsave(
  "insertion.gif",
  [imageio.imread(frame_dir / f"frame_{step:03d}.png") for step in range(num_steps)],
)

In [ ]:
imageio.mimsave(
  "insertion.gif",
  [imageio.imread(frame_dir / f"frame_{step:03d}.png") for step in range(num_steps)],
)